# scANVI Results Visualization - Debugging Version

**Purpose:** Comprehensive visualization of scANVI training results

**Version:** 1.0 - Debug-Enabled

**Date:** 2026-02-07

## Section 0: Imports and Configuration

In [ ]:
import sys
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from scipy.sparse import issparse

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

import scanpy as sc
import json

warnings.filterwarnings("ignore")
%matplotlib inline

print("✅ Imports successful!")

In [ ]:
# =====================================================
# USER CONFIGURATION - MODIFY HERE
# =====================================================

# Input/Output
INPUT_H5AD = "/home/h2048/data/py/0207/merged_scanvi_L2_prod_v1/merged_scanvi_L2_prod.h5ad"
OUTPUT_DIR = "/home/h2048/data/py/0208/merged_scanvi_L2_prod_v1/figures"

# Marker genes
MARKER_GENES = [
    "CD19", "MS4A1", "CD27", "IGHD", "IGHM", 
    "MZB1", "SDC1", "JCHAIN", "XBP1", "PRDM1"
]

# Figure settings
DPI = 300
FIG_FORMAT = "pdf"  # or "png"

# =====================================================
# COLUMN NAMES - AUTO-DETECT OR MODIFY IF NEEDED
# =====================================================
# Leave as None for auto-detection, or specify exact column names

DATASOURCE_KEY = None      # Auto: looks for 'data_source', 'source', 'batch'
PRED_LABEL_KEY = None      # Auto: looks for '*pred*', 'cell_type', 'celltype'
CONF_KEY = None            # Auto: looks for '*confidence*', '*conf*'
BATCH_KEY = None           # Auto: looks for 'sample', 'batch', 'donor'
TISSUE_KEY = None          # Auto: looks for 'tissue', 'organ'
UMAP_KEY = None            # Auto: looks for 'X_umap*'
LATENT_KEY = None          # Auto: looks for 'X_scANVI*', 'X_scvi*'

# Optional keys (used if present)
REF_LABEL_KEY = None       # Auto: looks for 'Cell_Type*', 'celltype*'
TRAIN_LABEL_KEY = None     # Auto: looks for '*train*'

print("✅ Configuration loaded!")

In [ ]:
# Plot settings
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
sc.settings.set_figure_params(dpi=DPI, facecolor='white', format=FIG_FORMAT, vector_friendly=False)

# Create output directory
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {output_dir}")
print("✅ Settings configured!")

## Section 1: Load Data and Inspect Structure

In [ ]:
# Load data
print(f"Loading: {INPUT_H5AD}")
adata = sc.read_h5ad(INPUT_H5AD)
print(f"Shape: {adata.shape}")
print("✅ Data loaded!")

In [ ]:
# =====================================================
# CRITICAL: Inspect your data structure
# =====================================================

print("=" * 80)
print("DATA STRUCTURE INSPECTION")
print("=" * 80)

print("\n📋 Available .obs columns:")
for i, col in enumerate(adata.obs.columns, 1):
    dtype = adata.obs[col].dtype
    n_unique = adata.obs[col].nunique()
    print(f"  {i:2d}. {col:40s} (dtype: {str(dtype):15s}, unique: {n_unique})")

print("\n📊 Available .obsm keys:")
for i, key in enumerate(adata.obsm.keys(), 1):
    shape = adata.obsm[key].shape
    print(f"  {i:2d}. {key:30s} (shape: {shape})")

print("\n🧬 Available .var columns:")
for i, col in enumerate(adata.var.columns, 1):
    print(f"  {i:2d}. {col}")

print("\n📦 Available .uns keys:")
for i, key in enumerate(adata.uns.keys(), 1):
    print(f"  {i:2d}. {key}")

## Section 2: Auto-Detect Column Names

**If auto-detection fails, manually set the column names in the configuration cell above.**

In [ ]:
def auto_detect_column(adata, candidates, location='obs', required=True, verbose=True):
    """
    Auto-detect column name from candidates.
    
    Parameters:
    -----------
    candidates : list of str
        List of possible column names (wildcards supported with *)
    location : str
        'obs' or 'obsm'
    required : bool
        If True, raise error if not found
    """
    if location == 'obs':
        available = adata.obs.columns
    elif location == 'obsm':
        available = list(adata.obsm.keys())
    else:
        raise ValueError(f"Unknown location: {location}")
    
    for candidate in candidates:
        if '*' in candidate:
            # Wildcard matching
            import fnmatch
            matches = [col for col in available if fnmatch.fnmatch(col.lower(), candidate.lower())]
            if matches:
                if verbose:
                    print(f"  ✓ Found '{matches[0]}' (pattern: {candidate})")
                return matches[0]
        else:
            # Exact match (case-insensitive)
            matches = [col for col in available if col.lower() == candidate.lower()]
            if matches:
                if verbose:
                    print(f"  ✓ Found '{matches[0]}'")
                return matches[0]
    
    if required:
        raise ValueError(f"Could not find any of {candidates} in {location}. Available: {list(available)[:10]}")
    
    if verbose:
        print(f"  ✗ Not found (optional): {candidates}")
    return None

print("=" * 80)
print("AUTO-DETECTING COLUMN NAMES")
print("=" * 80)

# Required columns
print("\n📍 Required columns:")
if PRED_LABEL_KEY is None:
    PRED_LABEL_KEY = auto_detect_column(
        adata, 
        ['L2_scanvi_pred', '*scanvi*pred*', '*pred*', 'predicted_labels', 'cell_type', 'celltype'],
        required=True
    )

if UMAP_KEY is None:
    UMAP_KEY = auto_detect_column(
        adata,
        ['X_umap_scanvi', 'X_umap*', 'umap'],
        location='obsm',
        required=True
    )

# Optional columns
print("\n📍 Optional columns:")
if DATASOURCE_KEY is None:
    DATASOURCE_KEY = auto_detect_column(
        adata,
        ['data_source', 'source', 'dataset', 'batch'],
        required=False
    )

if CONF_KEY is None:
    CONF_KEY = auto_detect_column(
        adata,
        ['L2_scanvi_confidence', '*confidence*', '*conf*', 'prediction_score'],
        required=False
    )

if BATCH_KEY is None:
    BATCH_KEY = auto_detect_column(
        adata,
        ['sample', 'batch', 'donor', 'patient'],
        required=False
    )

if TISSUE_KEY is None:
    TISSUE_KEY = auto_detect_column(
        adata,
        ['tissue', 'organ', 'site'],
        required=False
    )

if REF_LABEL_KEY is None:
    REF_LABEL_KEY = auto_detect_column(
        adata,
        ['Cell_Type_L2', 'Cell_Type*', 'celltype_ground_truth', 'ground_truth'],
        required=False
    )

if TRAIN_LABEL_KEY is None:
    TRAIN_LABEL_KEY = auto_detect_column(
        adata,
        ['Cell_Type_L2_train', '*train*', 'training_labels'],
        required=False
    )

if LATENT_KEY is None:
    LATENT_KEY = auto_detect_column(
        adata,
        ['X_scANVI_L2', 'X_scANVI*', 'X_scvi*', 'latent'],
        location='obsm',
        required=False
    )

print("\n✅ Column detection complete!")

In [ ]:
# Summary of detected columns
print("=" * 80)
print("DETECTED COLUMN MAPPING")
print("=" * 80)

column_map = {
    'Prediction Label': PRED_LABEL_KEY,
    'UMAP Coordinates': UMAP_KEY,
    'Data Source': DATASOURCE_KEY,
    'Confidence': CONF_KEY,
    'Batch': BATCH_KEY,
    'Tissue': TISSUE_KEY,
    'Reference Label': REF_LABEL_KEY,
    'Training Label': TRAIN_LABEL_KEY,
    'Latent Representation': LATENT_KEY,
}

for key, value in column_map.items():
    status = "✅" if value else "❌"
    print(f"{status} {key:25s}: {value}")

print("\n⚠️  If any required column is missing, manually set it in the configuration cell above!")

## Section 3: Helper Functions

In [ ]:
def save_rasterized_figure(fig, path, dpi=300):
    """Save figure with scatter plots rasterized."""
    for ax in fig.axes:
        for coll in ax.collections:
            coll.set_rasterized(True)
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✅ Saved: {path.name}")

def get_available_markers(adata, markers):
    """Get markers that exist in the data."""
    return [g for g in markers if g in adata.var_names]

def create_celltype_palette(categories):
    """Create fixed color palette for cell types."""
    n = len(categories)
    if n <= 20:
        colors = list(plt.get_cmap("tab20").colors)[:n]
    elif n <= 102:
        colors = sc.pl.palettes.default_102[:n]
    else:
        colors = [plt.cm.hsv(i / n) for i in range(n)]
    return dict(zip(categories, colors))

# Color schemes
PALETTE_SOURCE = {"reference": "#1f77b4", "query": "#ff7f0e"}
CMAP_CONFIDENCE = "viridis"
CMAP_EXPRESSION = "Reds"

print("✅ Helper functions defined!")

## Section 4: Data Summary

In [ ]:
print("=" * 80)
print("DATA SUMMARY")
print("=" * 80)

# Data source
if DATASOURCE_KEY:
    print(f"\nData source distribution:")
    for src, count in adata.obs[DATASOURCE_KEY].value_counts().items():
        pct = count / adata.n_obs * 100
        print(f"  {src}: {count:,} ({pct:.1f}%)")

# Predictions
n_types = adata.obs[PRED_LABEL_KEY].nunique()
print(f"\nPredicted cell types: {n_types}")
print(f"\nTop 10 predictions:")
for ct, count in adata.obs[PRED_LABEL_KEY].value_counts().head(10).items():
    pct = count / adata.n_obs * 100
    print(f"  {ct}: {count:,} ({pct:.1f}%)")

# Confidence
if CONF_KEY:
    conf = adata.obs[CONF_KEY]
    print(f"\nConfidence statistics:")
    print(f"  Mean: {conf.mean():.3f}")
    print(f"  Median: {conf.median():.3f}")
    print(f"  Q1: {conf.quantile(0.25):.3f}")
    print(f"  Q3: {conf.quantile(0.75):.3f}")
    print(f"  High (>0.9): {(conf > 0.9).sum():,} ({(conf > 0.9).mean()*100:.1f}%)")
    print(f"  Medium (0.5-0.9): {((conf >= 0.5) & (conf <= 0.9)).sum():,} ({((conf >= 0.5) & (conf <= 0.9)).mean()*100:.1f}%)")
    print(f"  Low (<0.5): {(conf < 0.5).sum():,} ({(conf < 0.5).mean()*100:.1f}%)")

# Markers
available_markers = get_available_markers(adata, MARKER_GENES)
print(f"\nMarker genes:")
print(f"  Requested: {len(MARKER_GENES)}")
print(f"  Available: {len(available_markers)}")
if len(available_markers) > 0:
    print(f"  Genes: {available_markers}")
else:
    print("  ⚠️  No markers found in data!")

## Section 5: Visualization 1 - Overview (6-panel)

In [ ]:
print("Creating Overview (6-panel)...")

fig, axes = plt.subplots(2, 3, figsize=(24, 16))

# Get cell type categories and palette
ct_categories = sorted(adata.obs[PRED_LABEL_KEY].unique())
ct_palette = create_celltype_palette(ct_categories)

# Panel 1: Data source
if DATASOURCE_KEY:
    sc.pl.umap(
        adata,
        color=DATASOURCE_KEY,
        ax=axes[0, 0],
        show=False,
        title="Data Source",
        palette=PALETTE_SOURCE,
        frameon=False,
        s=20
    )
else:
    axes[0, 0].text(0.5, 0.5, "Data source\nnot available", 
                   ha='center', va='center', transform=axes[0, 0].transAxes, fontsize=14)
    axes[0, 0].axis('off')

# Panel 2: Predicted cell types
sc.pl.umap(
    adata,
    color=PRED_LABEL_KEY,
    ax=axes[0, 1],
    show=False,
    title="Predicted Cell Types",
    legend_loc="right margin",
    palette=ct_palette,
    frameon=False,
    s=20
)

# Panel 3: Confidence
if CONF_KEY:
    sc.pl.umap(
        adata,
        color=CONF_KEY,
        ax=axes[0, 2],
        show=False,
        title="Mapping Confidence",
        cmap=CMAP_CONFIDENCE,
        vmin=0,
        vmax=1,
        frameon=False,
        s=20
    )
else:
    axes[0, 2].text(0.5, 0.5, "Confidence\nnot available", 
                   ha='center', va='center', transform=axes[0, 2].transAxes, fontsize=14)
    axes[0, 2].axis('off')

# Panel 4: Batch
if BATCH_KEY:
    sc.pl.umap(
        adata,
        color=BATCH_KEY,
        ax=axes[1, 0],
        show=False,
        title="Batch",
        frameon=False,
        s=20,
        legend_loc=None
    )
else:
    axes[1, 0].text(0.5, 0.5, "Batch\nnot available", 
                   ha='center', va='center', transform=axes[1, 0].transAxes, fontsize=14)
    axes[1, 0].axis('off')

# Panel 5: Tissue
if TISSUE_KEY:
    sc.pl.umap(
        adata,
        color=TISSUE_KEY,
        ax=axes[1, 1],
        show=False,
        title="Tissue",
        frameon=False,
        s=20,
        legend_loc="right margin"
    )
else:
    axes[1, 1].text(0.5, 0.5, "Tissue\nnot available", 
                   ha='center', va='center', transform=axes[1, 1].transAxes, fontsize=14)
    axes[1, 1].axis('off')

# Panel 6: First marker
if available_markers:
    marker = available_markers[0]
    sc.pl.umap(
        adata,
        color=marker,
        ax=axes[1, 2],
        show=False,
        title=f"{marker} Expression",
        cmap=CMAP_EXPRESSION,
        frameon=False,
        s=20
    )
else:
    axes[1, 2].text(0.5, 0.5, "No markers\navailable", 
                   ha='center', va='center', transform=axes[1, 2].transAxes, fontsize=14)
    axes[1, 2].axis('off')

plt.tight_layout()
output_path = output_dir / f"overview_6panel.{FIG_FORMAT}"
save_rasterized_figure(fig, output_path, dpi=DPI)

print("✅ Overview complete!")

## Section 6: Visualization 2 - Reference vs Query Comparison

In [ ]:
if DATASOURCE_KEY and REF_LABEL_KEY:
    print("Creating Reference vs Query Comparison...")
    
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    
    # Left: Reference ground truth
    ref_mask = adata.obs[DATASOURCE_KEY] == "reference"
    adata_ref = adata[ref_mask]
    
    sc.pl.umap(
        adata_ref,
        color=REF_LABEL_KEY,
        ax=axes[0],
        show=False,
        title="Reference: Ground Truth Labels",
        legend_loc="right margin",
        palette=ct_palette,
        frameon=False,
        s=30
    )
    
    # Right: Query predictions
    qry_mask = adata.obs[DATASOURCE_KEY] == "query"
    adata_qry = adata[qry_mask]
    
    sc.pl.umap(
        adata_qry,
        color=PRED_LABEL_KEY,
        ax=axes[1],
        show=False,
        title="Query: Predicted Labels",
        legend_loc="right margin",
        palette=ct_palette,
        frameon=False,
        s=30
    )
    
    plt.tight_layout()
    output_path = output_dir / f"reference_vs_query_comparison.{FIG_FORMAT}"
    save_rasterized_figure(fig, output_path, dpi=DPI)
    
    print("✅ Reference vs Query complete!")
else:
    print("⚠️  Skipping Reference vs Query (missing required columns)")

## Section 7: Visualization 3 - Marker Expression Panel

In [ ]:
if available_markers:
    print(f"Creating Marker Expression Panel ({len(available_markers)} markers)...")
    
    n_markers = len(available_markers)
    ncols = 4
    nrows = int(np.ceil(n_markers / ncols))
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(20, 5*nrows))
    if nrows == 1:
        axes = axes.reshape(1, -1)
    
    for i, marker in enumerate(available_markers):
        row = i // ncols
        col = i % ncols
        
        sc.pl.umap(
            adata,
            color=marker,
            ax=axes[row, col],
            show=False,
            title=f"{marker}",
            cmap=CMAP_EXPRESSION,
            frameon=False,
            s=30
        )
    
    # Hide unused subplots
    for i in range(n_markers, nrows * ncols):
        row = i // ncols
        col = i % ncols
        axes[row, col].axis('off')
    
    plt.tight_layout()
    output_path = output_dir / f"marker_expression_panel.{FIG_FORMAT}"
    save_rasterized_figure(fig, output_path, dpi=DPI)
    
    print("✅ Marker panel complete!")
else:
    print("⚠️  Skipping Marker Expression Panel (no markers available)")

## Section 8: Visualization 4 - Cell Type Distribution

In [ ]:
print("Creating Cell Type Distribution...")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar plot
ct_counts = adata.obs[PRED_LABEL_KEY].value_counts()
ct_sorted = ct_counts.sort_values(ascending=True)

axes[0].barh(range(len(ct_sorted)), ct_sorted.values, color='steelblue')
axes[0].set_yticks(range(len(ct_sorted)))
axes[0].set_yticklabels(ct_sorted.index)
axes[0].set_xlabel('Number of Cells', fontsize=12)
axes[0].set_title('Cell Type Distribution', fontsize=14, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Pie chart
axes[1].pie(
    ct_counts.values[:10],
    labels=ct_counts.index[:10],
    autopct='%1.1f%%',
    startangle=90
)
axes[1].set_title('Cell Type Proportions (Top 10)', fontsize=14, fontweight='bold')

plt.tight_layout()
output_path = output_dir / f"celltype_distribution.{FIG_FORMAT}"
plt.savefig(output_path, dpi=DPI, bbox_inches='tight')
plt.close(fig)
print(f"  ✅ Saved: {output_path.name}")

## Section 9: Visualization 5 - Confidence Analysis

In [ ]:
if CONF_KEY:
    print("Creating Confidence Analysis...")
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    conf = adata.obs[CONF_KEY]
    
    # Panel 1: Histogram
    axes[0, 0].hist(conf, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    axes[0, 0].axvline(0.9, color='red', linestyle='--', linewidth=2, label='High threshold (0.9)')
    axes[0, 0].axvline(0.5, color='orange', linestyle='--', linewidth=2, label='Medium threshold (0.5)')
    axes[0, 0].set_xlabel('Confidence Score', fontsize=12)
    axes[0, 0].set_ylabel('Number of Cells', fontsize=12)
    axes[0, 0].set_title('Confidence Distribution', fontsize=14, fontweight='bold')
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)
    axes[0, 0].spines['top'].set_visible(False)
    axes[0, 0].spines['right'].set_visible(False)
    
    # Panel 2: Confidence by cell type (violin)
    ct_conf_df = pd.DataFrame({
        'cell_type': adata.obs[PRED_LABEL_KEY],
        'confidence': conf
    })
    
    # Sort by median confidence
    ct_order = ct_conf_df.groupby('cell_type')['confidence'].median().sort_values(ascending=False).index[:15]
    ct_conf_df_top = ct_conf_df[ct_conf_df['cell_type'].isin(ct_order)]
    
    sns.violinplot(
        data=ct_conf_df_top,
        x='cell_type',
        y='confidence',
        order=ct_order,
        ax=axes[0, 1],
        palette='Set2'
    )
    axes[0, 1].axhline(0.9, color='red', linestyle='--', linewidth=1, alpha=0.5)
    axes[0, 1].axhline(0.5, color='orange', linestyle='--', linewidth=1, alpha=0.5)
    axes[0, 1].set_xlabel('Cell Type', fontsize=12)
    axes[0, 1].set_ylabel('Confidence', fontsize=12)
    axes[0, 1].set_title('Confidence by Cell Type (Top 15)', fontsize=14, fontweight='bold')
    axes[0, 1].tick_params(axis='x', rotation=45)
    axes[0, 1].grid(axis='y', alpha=0.3)
    
    # Panel 3: Confidence categories pie
    conf_cats = pd.cut(
        conf,
        bins=[0, 0.5, 0.9, 1.0],
        labels=['Low (<0.5)', 'Medium (0.5-0.9)', 'High (>0.9)']
    )
    conf_cat_counts = conf_cats.value_counts()
    
    axes[1, 0].pie(
        conf_cat_counts.values,
        labels=conf_cat_counts.index,
        autopct='%1.1f%%',
        colors=['#ff6b6b', '#ffd93d', '#6bcf7f'],
        startangle=90
    )
    axes[1, 0].set_title('Confidence Categories', fontsize=14, fontweight='bold')
    
    # Panel 4: Box plot by cell type
    sns.boxplot(
        data=ct_conf_df_top,
        x='cell_type',
        y='confidence',
        order=ct_order,
        ax=axes[1, 1],
        palette='Set2'
    )
    axes[1, 1].axhline(0.9, color='red', linestyle='--', linewidth=1, alpha=0.5)
    axes[1, 1].axhline(0.5, color='orange', linestyle='--', linewidth=1, alpha=0.5)
    axes[1, 1].set_xlabel('Cell Type', fontsize=12)
    axes[1, 1].set_ylabel('Confidence', fontsize=12)
    axes[1, 1].set_title('Confidence Distribution by Cell Type (Top 15)', fontsize=14, fontweight='bold')
    axes[1, 1].tick_params(axis='x', rotation=45)
    axes[1, 1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    output_path = output_dir / f"confidence_analysis.{FIG_FORMAT}"
    plt.savefig(output_path, dpi=DPI, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✅ Saved: {output_path.name}")
else:
    print("⚠️  Skipping Confidence Analysis (confidence column not available)")

## Section 10: Visualization 6 - Training vs Prediction Comparison

In [ ]:
if TRAIN_LABEL_KEY:
    print("Creating Training vs Prediction Comparison...")
    
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    
    # Left: Training labels
    sc.pl.umap(
        adata,
        color=TRAIN_LABEL_KEY,
        ax=axes[0],
        show=False,
        title="Training Labels (Ref + High-Conf Query)",
        legend_loc="right margin",
        palette=ct_palette,
        frameon=False,
        s=25
    )
    
    # Right: Predictions
    sc.pl.umap(
        adata,
        color=PRED_LABEL_KEY,
        ax=axes[1],
        show=False,
        title="scANVI Predictions",
        legend_loc="right margin",
        palette=ct_palette,
        frameon=False,
        s=25
    )
    
    plt.tight_layout()
    output_path = output_dir / f"training_vs_prediction.{FIG_FORMAT}"
    save_rasterized_figure(fig, output_path, dpi=DPI)
    
    print("✅ Training vs Prediction complete!")
else:
    print("⚠️  Skipping Training vs Prediction (training label column not available)")

## Section 11: Visualization 7 - High-Resolution Exports

In [ ]:
print("Creating High-Resolution Individual Exports...")

export_params = {'frameon': False, 's': 50, 'show': False}

# 1. Cell types (on data legend)
fig, ax = plt.subplots(figsize=(10, 8))
sc.pl.umap(
    adata,
    color=PRED_LABEL_KEY,
    ax=ax,
    title="",
    legend_loc="on data",
    legend_fontsize=10,
    legend_fontoutline=2,
    palette=ct_palette,
    **export_params
)
output_path = output_dir / f"umap_celltypes_highres.{FIG_FORMAT}"
save_rasterized_figure(fig, output_path, dpi=DPI)

# 2. Confidence
if CONF_KEY:
    fig, ax = plt.subplots(figsize=(10, 8))
    sc.pl.umap(
        adata,
        color=CONF_KEY,
        ax=ax,
        title="",
        cmap=CMAP_CONFIDENCE,
        vmin=0,
        vmax=1,
        **export_params
    )
    output_path = output_dir / f"umap_confidence_highres.{FIG_FORMAT}"
    save_rasterized_figure(fig, output_path, dpi=DPI)

# 3. Data source
if DATASOURCE_KEY:
    fig, ax = plt.subplots(figsize=(10, 8))
    sc.pl.umap(
        adata,
        color=DATASOURCE_KEY,
        ax=ax,
        title="",
        palette=PALETTE_SOURCE,
        **export_params
    )
    output_path = output_dir / f"umap_datasource_highres.{FIG_FORMAT}"
    save_rasterized_figure(fig, output_path, dpi=DPI)

print("✅ High-res exports complete!")

## Section 12: Save Summary Statistics

In [ ]:
print("Saving summary statistics...")

summary_stats = {
    'timestamp': datetime.now().isoformat(),
    'input_file': str(INPUT_H5AD),
    'n_cells': int(adata.n_obs),
    'n_genes': int(adata.n_vars),
    'cell_types': adata.obs[PRED_LABEL_KEY].value_counts().to_dict()
}

if DATASOURCE_KEY:
    summary_stats['data_sources'] = adata.obs[DATASOURCE_KEY].value_counts().to_dict()

if CONF_KEY:
    conf = adata.obs[CONF_KEY]
    summary_stats['confidence'] = {
        'mean': float(conf.mean()),
        'median': float(conf.median()),
        'std': float(conf.std()),
        'min': float(conf.min()),
        'max': float(conf.max()),
        'high_gt_0.9': int((conf > 0.9).sum()),
        'medium_0.5_0.9': int(((conf >= 0.5) & (conf <= 0.9)).sum()),
        'low_lt_0.5': int((conf < 0.5).sum())
    }

# Save summary
summary_path = output_dir / "visualization_summary.json"
with open(summary_path, 'w') as f:
    json.dump(summary_stats, f, indent=2)
print(f"✅ Summary saved: {summary_path}")

# Save cell type counts to CSV
ct_counts_df = pd.DataFrame({
    'cell_type': adata.obs[PRED_LABEL_KEY].value_counts().index,
    'count': adata.obs[PRED_LABEL_KEY].value_counts().values,
    'percentage': (adata.obs[PRED_LABEL_KEY].value_counts().values / adata.n_obs * 100)
})
ct_counts_path = output_dir / "celltype_counts.csv"
ct_counts_df.to_csv(ct_counts_path, index=False)
print(f"✅ Cell type counts saved: {ct_counts_path}")

## Section 13: Final Summary

In [ ]:
print("\n" + "=" * 80)
print("VISUALIZATION COMPLETE!")
print("=" * 80)

print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"\nGenerated files:")

output_files = sorted(output_dir.glob(f"*.{FIG_FORMAT}"))
for f in output_files:
    file_size = f.stat().st_size / 1024  # KB
    print(f"  - {f.name} ({file_size:.1f} KB)")

print(f"\nAdditional files:")
print(f"  - visualization_summary.json")
print(f"  - celltype_counts.csv")

print("\n" + "=" * 80)
print("All visualizations saved successfully!")
print("=" * 80)